# 01 Signal model and synthetic reflectivity profiles

This notebook confirms the elevation signal model that the physics check operates on. `PhysicsQuantitiesCheck` compares two reflectivity profiles per pixel along the elevation axis: a Gaussian-mixture fit and a Capon tomogram. Both are non-negative power spectra sampled on a shared height axis `x`.

Modules exercised: `tools.gaussians.GaussianMixture`, and the tensor layout helper `PhysicsQuantitiesCheck._to_tensor`, which we replicate as `to_check_tensor`.

We synthesize single- and multi-component scatterer profiles from known parameters and verify that the discretised mass, mean, and shape match the analytic expectation.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from tools.gaussians import GaussianMixture

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


## Height axis and a single Gaussian scatterer

A single elevated scatterer is a Gaussian in height. We place it at `mu = 12 m` with spread `sigma = 3 m` and unit amplitude, on a height axis spanning the canopy volume.

In [ ]:
x_axis = np.linspace(-20.0, 40.0, 160).astype(np.float32)
dx     = float(x_axis[1] - x_axis[0])

single = gaussian_profile(x_axis, amp=1.0, mu=12.0, sigma=3.0)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(x_axis, single, color="C0")
ax.axvline(12.0, color="0.4", ls="--", lw=0.9, label="mu = 12 m")
ax.set_xlabel("elevation [m]")
ax.set_ylabel("reflectivity")
ax.set_title("Single-scatterer elevation profile")
ax.legend()
plt.show()

## Multi-component profile (ground plus canopy)

A vegetated pixel has a strong ground return plus a broader canopy lobe. We build a two-component mixture and overlay the individual components.

In [ ]:
amps  = [1.0, 0.5]
mus   = [0.0, 18.0]
sigs  = [2.0, 6.0]

mixture = gaussian_profile(x_axis, amps, mus, sigs)
comp0   = gaussian_profile(x_axis, amps[0], mus[0], sigs[0])
comp1   = gaussian_profile(x_axis, amps[1], mus[1], sigs[1])

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(x_axis, mixture, color="k",  lw=1.8, label="mixture")
ax.plot(x_axis, comp0,   color="C0", lw=1.0, ls="--", label="ground")
ax.plot(x_axis, comp1,   color="C1", lw=1.0, ls="--", label="canopy")
ax.set_xlabel("elevation [m]")
ax.set_ylabel("reflectivity")
ax.set_title("Two-component ground-plus-canopy profile")
ax.legend()
plt.show()

## Tensor layout used by the check

Inside `check.py` every profile batch is reshaped to `(1, n_bins, 1, n_pixels)`: a single batch, the elevation samples on the channel axis, and pixels on the spatial width axis. We confirm the helper produces this layout and that a round trip preserves values.

In [ ]:
batch = np.stack([single, mixture], axis=0)   # (n_pixels=2, n_bins)
tens  = to_check_tensor(batch)

print("profiles array shape:", batch.shape)
print("check tensor shape  :", tuple(tens.shape))

round_trip = tens[0, :, 0, :].T.cpu().numpy()
print("max round-trip error:", float(np.abs(round_trip - batch).max()))

## Expected outcome

The single profile is a clean bell centred on its mean; the mixture shows two resolvable lobes whose sum equals the overlaid components. The check tensor has shape `(1, n_bins, 1, n_pixels)` and the round-trip error is zero, confirming the layout convention the downstream physics terms rely on.